# NB2 — Pondération, caractères et hybrides avec tuning intégré

Ce notebook couvre `P06` à `P10`. Il conserve le benchmark initial, puis ajoute une phase de tuning accéléré des algorithmes sur des données déjà vectorisées une seule fois par pipeline.

Ce notebook conserve la **phase baseline sans optimisation**, puis ajoute une **phase d’optimisation accélérée**.
Le principe retenu est le suivant : pour chaque pipeline, on ajuste d’abord le **préprocesseur / vectoriseur une seule fois** sur un sous-ensemble d’apprentissage, puis on teste plusieurs réglages du **classifieur uniquement** sur les mêmes données déjà transformées. Cela réduit fortement le temps d’exécution.

Cette stratégie est très pratique pour explorer rapidement des réglages d’algorithmes, mais il faut bien comprendre qu’elle constitue une **optimisation accélérée**, plus pragmatique qu’une recherche entièrement relancée sur tout le pipeline à chaque itération.


In [ ]:
# Pour un run sur Colab

'''
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Disaster-Tweets-NLP"
MODELS_DIR = f"{PROJECT_ROOT}/notebooks/models_training"

%cd "{MODELS_DIR}"

import sys
if MODELS_DIR not in sys.path:
    sys.path.append(MODELS_DIR)

print("Projet :", PROJECT_ROOT)
print("Dossier courant :", MODELS_DIR)
'''


'\nfrom google.colab import drive\ndrive.mount(\'/content/drive\')\n\nPROJECT_ROOT = "/content/drive/MyDrive/Disaster-Tweets-NLP"\nMODELS_DIR = f"{PROJECT_ROOT}/notebooks/models_training"\n\n%cd "{MODELS_DIR}"\n\nimport sys\nif MODELS_DIR not in sys.path:\n    sys.path.append(MODELS_DIR)\n\nprint("Projet :", PROJECT_ROOT)\nprint("Dossier courant :", MODELS_DIR)\n'

In [ ]:
# Installation éventuelle (décommente si nécessaire)
# !pip install pandas numpy scikit-learn scipy matplotlib gensim sentence-transformers openpyxl mlflow

import json
from collections import OrderedDict
from pathlib import Path

import pandas as pd
from mlflow_utils import (
    fit_evaluate_and_log_sklearn_pipeline,
    setup_mlflow_tracking,
)
from nlp_disaster_utils import (
    load_train_test_xy,
    round_results,
    save_results_bundle,
    seed_everything,
    stratified_validation_split,
)
from pipeline_tuning_utils import (
    compare_baseline_vs_tuned,
    evaluate_refit_outputs,
    fit_transform_preprocessor_once,
    log_tuning_run_to_mlflow,
    safe_scores,
    split_pipeline_preprocessor_estimator,
    tune_classifier_on_fixed_features,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC


seed_everything(42)


C:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\_internal\_fields.py:161: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [ ]:
DATA_DIR = "../../data/processed_data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"

TEXT_COL = "text"
LABEL_COL = "target"
USE_AUX_TEXT_COLUMNS = False
LOWERCASE_TEXT = False
RANDOM_STATE = 42

OUTPUT_STEM = "NB2_weighting_char_hybrid"
RESULTS_DIR = "../../outputs/NB2"
TUNING_OUTPUT_DIR = Path(RESULTS_DIR) / "tuning"
TUNING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Configuration MLflow
MLFLOW_EXPERIMENT_NAME = "DT_NB2_weighting_char_hybrid"
MLFLOW_TRACKING_URI = Path("../../outputs/mlruns").resolve().as_uri()
MLFLOW_LOG_MODEL = False
USE_MLFLOW = True

tracking_uri = setup_mlflow_tracking(
    experiment_name=MLFLOW_EXPERIMENT_NAME,
    tracking_uri=MLFLOW_TRACKING_URI,
)
print("MLflow tracking URI :", tracking_uri)
print("MLflow experiment   :", MLFLOW_EXPERIMENT_NAME)

# Paramètres de tuning accéléré
VAL_SIZE_FOR_TUNING = 0.15
PRIMARY_TUNING_METRIC = "f1_pos"


MLflow tracking URI : file:///C:/Users/DELL/Documents/Classes/ISE2/ISE2_2026/SEM2/ML2/Projet/Disaster-Tweets-NLP/outputs/mlruns
MLflow experiment   : DT_NB2_weighting_char_hybrid


C:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Même configuration que le NB1, avec cette fois l'expérience MLflow nommée `DT_NB2_weighting_char_hybrid` et les résultats sauvegardés dans `outputs/NB2`. Le reste est identique : 15% du train réservés pour la validation du tuning, et le **F1 de la classe 1** comme métrique de référence pour sélectionner les meilleurs hyperparamètres.

In [ ]:
df_train, X_train, y_train, df_test, X_test, y_test = load_train_test_xy(
    train_path=TRAIN_PATH,
    test_path=TEST_PATH,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    use_extra_cols=USE_AUX_TEXT_COLUMNS,
    lowercase=LOWERCASE_TEXT,
)

print("Taille train :", len(X_train))
print("Taille test  :", len(X_test))
print("\nDistribution des classes - train :")
print(y_train.value_counts(normalize=True).sort_index())
print("\nDistribution des classes - test :")
print(y_test.value_counts(normalize=True).sort_index())


Taille train : 9096
Taille test  : 2274

Distribution des classes - train :
target
0    0.814094
1    0.185906
Name: proportion, dtype: float64

Distribution des classes - test :
target
0    0.813984
1    0.186016
Name: proportion, dtype: float64


Les données chargées sont identiques au NB1 : **9 096 tweets** en entraînement et **2 274** en test, avec le même déséquilibre de classes (~81% Non-Disaster / ~19% Disaster) dans les deux sets. La base de comparaison est donc la même, ce qui permettra d'évaluer proprement l'apport des nouvelles représentations textuelles.

In [ ]:
word_tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
)
char_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
)

pipelines = OrderedDict({
    "P06_TFIDF_Sublinear_LogReg": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)),
        ("clf", LogisticRegression(max_iter=2500, C=1.0)),
    ]),
    "P07_CharTFIDF_LogReg": Pipeline([
        ("vect", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2)),
        ("clf", LogisticRegression(max_iter=2500, C=1.0)),
    ]),
    "P08_CharTFIDF_LinearSVC": Pipeline([
        ("vect", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2)),
        ("clf", LinearSVC(C=1.0)),
    ]),
    "P09_WordChar_Hybrid_LogReg": Pipeline([
        ("features", FeatureUnion([
            ("word_tfidf", word_tfidf),
            ("char_tfidf", char_tfidf),
        ])),
        ("clf", LogisticRegression(max_iter=3000, C=1.0)),
    ]),
    "P10_WordChar_Hybrid_LinearSVC": Pipeline([
        ("features", FeatureUnion([
            ("word_tfidf", word_tfidf),
            ("char_tfidf", char_tfidf),
        ])),
        ("clf", LinearSVC(C=1.0)),
    ]),
})


On définit ici les 5 nouveaux pipelines (P06 à P10) :

- **P06 (TF-IDF Sublinear + LogReg)** : même vectoriseur que P03 du NB1, mais avec `sublinear_tf=True` qui applique un log sur les fréquences des mots. Cela réduit l'influence des termes très répétitifs et donne plus de poids aux mots rares mais informatifs.
- **P07 (Char TF-IDF + LogReg)** : on travaille cette fois au niveau des caractères avec des séquences de 3 à 5 caractères (`char_wb`). L'option `char_wb` s'arrête aux frontières des mots, ce qui évite de créer des n-grammes qui chevauchent deux mots différents.
- **P08 (Char TF-IDF + LinearSVC)** : même représentation caractères que P07, mais avec une SVM linéaire comme classifieur.
- **P09 (Hybride Word+Char + LogReg)** : on concatène les deux vecteurs — mots et caractères — via un `FeatureUnion`. Le modèle bénéficie ainsi des deux types d'information en même temps.
- **P10 (Hybride Word+Char + LinearSVC)** : même hybride que P09 mais avec LinearSVC, qui est généralement plus rapide et parfois plus précis sur des espaces de features très denses comme celui-ci.

## Phase 1 — Baselines sans optimisation

Cette première phase reproduit le benchmark initial : chaque pipeline est exécuté tel quel, avec ses paramètres de départ.

In [ ]:
resultats = []
baseline_failures = []

for nom_pipeline, pipeline in pipelines.items():
    print(f"Entraînement baseline -> {nom_pipeline}")
    display(pipeline)
    print("-" * 80)

    try:
        metrics = fit_evaluate_and_log_sklearn_pipeline(
            name=nom_pipeline,
            estimator=pipeline,
            X_train=X_train,
            X_test=X_test,
            y_train=y_train,
            y_test=y_test,
            notebook_name="WEIGHTING",
            family_name="weighting_char_hybrid",
            output_dir=RESULTS_DIR,
            log_model=MLFLOW_LOG_MODEL,
        )
        resultats.append(metrics)
    except Exception as exc:
        baseline_failures.append({"pipeline": nom_pipeline, "error": str(exc)})
        print(f"Échec baseline pour {nom_pipeline} : {exc}")

baseline_df = round_results(pd.DataFrame(resultats))
display(baseline_df)

if baseline_failures:
    print("\nPipelines baseline en échec :")
    display(pd.DataFrame(baseline_failures))


Entraînement baseline -> P06_TFIDF_Sublinear_LogReg


Pipeline(steps=[('vect',
                 TfidfVectorizer(max_df=0.95, min_df=2, ngram_range=(1, 2),
                                 sublinear_tf=True)),
                ('clf', LogisticRegression(max_iter=2500))])

--------------------------------------------------------------------------------
Entraînement baseline -> P07_CharTFIDF_LogReg


Pipeline(steps=[('vect',
                 TfidfVectorizer(analyzer='char_wb', min_df=2,
                                 ngram_range=(3, 5))),
                ('clf', LogisticRegression(max_iter=2500))])

--------------------------------------------------------------------------------
Entraînement baseline -> P08_CharTFIDF_LinearSVC


...

Pipeline(steps=[('features',
                 FeatureUnion(transformer_list=[('word_tfidf',
                                                 TfidfVectorizer(max_df=0.95,
                                                                 min_df=2,
                                                                 ngram_range=(1,
                                                                              2))),
                                                ('char_tfidf',
                                                 TfidfVectorizer(analyzer='char_wb',
                                                                 min_df=2,
                                                                 ngram_range=(3,
                                                                              5)))])),
                ('clf', LogisticRegression(max_iter=3000))])

--------------------------------------------------------------------------------
Entraînement baseline -> P10_WordChar_Hybrid_LinearSVC


Pipeline(steps=[('features',
                 FeatureUnion(transformer_list=[('word_tfidf',
                                                 TfidfVectorizer(max_df=0.95,
                                                                 min_df=2,
                                                                 ngram_range=(1,
                                                                              2))),
                                                ('char_tfidf',
                                                 TfidfVectorizer(analyzer='char_wb',
                                                                 min_df=2,
                                                                 ngram_range=(3,
                                                                              5)))])),
                ('clf', LinearSVC())])

--------------------------------------------------------------------------------


pipeline,P06_TFIDF_Sublinear_LogReg,P07_CharTFIDF_LogReg,P08_CharTFIDF_LinearSVC,P09_WordChar_Hybrid_LogReg,P10_WordChar_Hybrid_LinearSVC
train_accuracy,0.8989,0.9070,0.9919,0.9394,0.9989
train_precision_macro,0.9372,0.9341,0.9929,0.9594,0.9986
train_recall_macro,0.7309,0.7565,0.9802,0.8407,0.9977
train_f1_macro,0.7858,0.8098,0.9864,0.8858,0.9982
train_precision_weighted,0.9080,0.9128,0.9919,0.9425,0.9989
train_recall_weighted,0.8989,0.9070,0.9919,0.9394,0.9989
train_f1_weighted,0.8836,0.8952,0.9918,0.9350,0.9989
train_precision_class_0,0.8907,0.9003,0.9913,0.9325,0.9991
train_recall_class_0,0.9982,0.9961,0.9988,0.9978,0.9996
train_f1_class_0,0.9414,0.9458,0.9950,0.9641,0.9993


Les 5 pipelines ont tourné sans erreur. En se concentrant sur les résultats **test**, voici ce qu'on retient :

Sur la **classe 1 (Disaster)**, notre cible principale :
- **P10 (Hybride + LinearSVC)** domine sur presque toutes les métriques globales : meilleure accuracy (0.892), meilleur F1 macro (0.807), meilleur recall classe 1 (0.617) et meilleur F1 classe 1 (0.680). La combinaison mots + caractères avec LinearSVC semble tirer le meilleur des deux représentations.
- **P06 (Sublinear + LogReg)** obtient la meilleure précision classe 1 (0.896) et le meilleur recall classe 0 (0.991), mais son recall sur la classe disaster chute à 0.345 — il est très prudent et rate beaucoup de vrais tweets de catastrophe.
- **P08 (Char + LinearSVC)** se distingue avec le meilleur recall classe 1 parmi les pipelines non-hybrides (0.575) et une bonne balanced accuracy (0.764), confirmant que la représentation caractères est plus sensible à la classe minoritaire.
- **P09 (Hybride + LogReg)** offre le meilleur ROC-AUC (0.923) et PR-AUC (0.781), ce qui en fait un bon candidat si on cherche à optimiser le ranking plutôt que le seuil de décision.

⚠️ L'overfitting reste présent sur P08 et P10 (train_f1_class_1 ≈ 0.98 vs test ≈ 0.65-0.68). Le tuning devrait aider à corriger cela, notamment en jouant sur la régularisation.

In [ ]:
save_results_bundle(pd.DataFrame(resultats), output_dir=RESULTS_DIR, stem=OUTPUT_STEM)
print(f"Fichiers CSV/XLSX baseline enregistrés dans {RESULTS_DIR}")


Fichiers CSV/XLSX baseline enregistrés dans ../../outputs/NB2


Les résultats de la phase baseline sont sauvegardés dans `outputs/NB2`, prêts à être comparés avec les résultats du tuning en fin de notebook.

## Phase 2 — Tuning accéléré avec vectorisation unique par pipeline

Ici, pour chaque pipeline, on sépare le **préprocesseur** du **classifieur**. On ajuste le préprocesseur **une seule fois** sur un sous-ensemble d’apprentissage, puis on teste différentes combinaisons d’hyperparamètres du classifieur sur les mêmes données déjà vectorisées. Enfin, on réajuste le meilleur classifieur sur tout le train transformé une seule fois et on l’évalue sur train et test.

In [ ]:
X_fit, X_val, y_fit, y_val = stratified_validation_split(
    X_train,
    y_train,
    val_size=VAL_SIZE_FOR_TUNING,
    random_state=RANDOM_STATE,
)

print("Taille tuning-fit :", len(X_fit))
print("Taille tuning-val :", len(X_val))


Taille tuning-fit : 7731
Taille tuning-val : 1365


Même découpage que dans le NB1 : **7 731 tweets** pour entraîner le vectoriseur et les classifieurs, et **1 365 tweets** mis de côté pour évaluer les combinaisons d'hyperparamètres. Le split stratifié garantit que les proportions des classes sont préservées dans les deux sous-ensembles.

In [ ]:
classifier_param_grids = {
    "P06_TFIDF_Sublinear_LogReg": {
        "C": [0.25, 0.5, 1.0, 2.0, 4.0],
        "class_weight": [None, "balanced"],
    },
    "P07_CharTFIDF_LogReg": {
        "C": [0.25, 0.5, 1.0, 2.0, 4.0],
        "class_weight": [None, "balanced"],
    },
    "P08_CharTFIDF_LinearSVC": {
        "C": [0.25, 0.5, 1.0, 2.0, 4.0],
        "class_weight": [None, "balanced"],
    },
    "P09_WordChar_Hybrid_LogReg": {
        "C": [0.25, 0.5, 1.0, 2.0, 4.0],
        "class_weight": [None, "balanced"],
    },
    "P10_WordChar_Hybrid_LinearSVC": {
        "C": [0.25, 0.5, 1.0, 2.0, 4.0],
        "class_weight": [None, "balanced"],
    },
}


Les grilles d'hyperparamètres sont identiques pour les 5 pipelines : on fait varier `C` sur 5 valeurs (de 0.25 à 4.0) et on teste avec ou sans `class_weight='balanced'`. C'est simple mais efficace — après les enseignements du NB1, on sait déjà que le rééquilibrage des classes a un impact significatif, donc on le garde systématiquement dans la grille pour tous les pipelines.

In [ ]:
tuning_rows = []
tuning_failures = []
tuned_metrics_rows = []

for nom_pipeline, pipeline in pipelines.items():
    print("=" * 100)
    print(f"Tuning accéléré -> {nom_pipeline}")

    param_grid = classifier_param_grids.get(nom_pipeline)
    if param_grid is None:
        tuning_failures.append({"pipeline": nom_pipeline, "error": "Grille d'hyperparamètres absente"})
        print("Aucune grille trouvée.")
        continue

    try:
        preprocessor, clf_name, base_estimator = split_pipeline_preprocessor_estimator(pipeline)

        transformed = fit_transform_preprocessor_once(
            preprocessor=preprocessor,
            X_fit=X_fit,
            y_fit=y_fit,
            X_val=X_val,
        )

        best_params, tuning_results_df = tune_classifier_on_fixed_features(
            base_estimator=base_estimator,
            param_grid=param_grid,
            X_fit=transformed["X_fit_transformed"],
            y_fit=y_fit,
            X_val=transformed["X_val_transformed"],
            y_val=y_val,
            primary_metric=PRIMARY_TUNING_METRIC,
        )

        tuning_results_df.insert(0, "pipeline", nom_pipeline)
        tuning_results_df.insert(1, "classifier_name", clf_name)

        best_preprocessor_full, _, best_estimator_template = split_pipeline_preprocessor_estimator(pipeline)
        best_preprocessor_full.fit(X_train, y_train)
        X_train_vec = best_preprocessor_full.transform(X_train)
        X_test_vec = best_preprocessor_full.transform(X_test)

        best_estimator = base_estimator.set_params(**best_params)
        best_estimator.fit(X_train_vec, y_train)

        train_pred = best_estimator.predict(X_train_vec)
        test_pred = best_estimator.predict(X_test_vec)
        train_score = safe_scores(best_estimator, X_train_vec)
        test_score = safe_scores(best_estimator, X_test_vec)

        final_metrics = evaluate_refit_outputs(
            pipeline_name=nom_pipeline,
            y_train=y_train,
            y_test=y_test,
            train_pred=train_pred,
            test_pred=test_pred,
            train_score=train_score,
            test_score=test_score,
        )
        final_metrics["best_params"] = json.dumps(best_params, ensure_ascii=False)
        final_metrics["best_val_primary_score"] = float(tuning_results_df.iloc[0]["primary_score"])
        final_metrics["best_val_f1_class_1"] = float(tuning_results_df.iloc[0]["val_f1_class_1"])
        final_metrics["best_val_recall_class_1"] = float(tuning_results_df.iloc[0]["val_recall_class_1"])
        final_metrics["best_val_f1_macro"] = float(tuning_results_df.iloc[0]["val_f1_macro"])
        final_metrics["best_val_balanced_accuracy"] = float(tuning_results_df.iloc[0]["val_balanced_accuracy"])

        tuning_rows.append(tuning_results_df.iloc[0].to_dict() | {
            "pipeline": nom_pipeline,
            "best_params": json.dumps(best_params, ensure_ascii=False),
        })
        tuned_metrics_rows.append(final_metrics)

        tuning_results_path = TUNING_OUTPUT_DIR / f"{nom_pipeline}_tuning_validation_results.csv"
        tuning_results_df.to_csv(tuning_results_path, index=False)

        if USE_MLFLOW:
            log_tuning_run_to_mlflow(
                run_name=nom_pipeline,
                notebook_name="WEIGHTING",
                family_name="weighting_char_hybrid",
                best_params=best_params,
                tuning_results_df=tuning_results_df,
                final_metrics=final_metrics,
                output_dir=TUNING_OUTPUT_DIR,
            )

        print("Meilleurs paramètres :", best_params)
        print("Meilleur score de validation :", tuning_results_df.iloc[0]["primary_score"])

    except Exception as exc:
        tuning_failures.append({"pipeline": nom_pipeline, "error": str(exc)})
        print(f"Échec tuning pour {nom_pipeline} : {exc}")


Tuning accéléré -> P06_TFIDF_Sublinear_LogReg
Meilleurs paramètres : {'C': 2.0, 'class_weight': 'balanced'}
Meilleur score de validation : 0.6859205776173285
Tuning accéléré -> P07_CharTFIDF_LogReg
Meilleurs paramètres : {'C': 1.0, 'class_weight': 'balanced'}
Meilleur score de validation : 0.6903553299492385
Tuning accéléré -> P08_CharTFIDF_LinearSVC
Meilleurs paramètres : {'C': 4.0, 'class_weight': None}
Meilleur score de validation : 0.6972111553784861
Tuning accéléré -> P09_WordChar_Hybrid_LogReg
Meilleurs paramètres : {'C': 4.0, 'class_weight': 'balanced'}
Meilleur score de validation : 0.7091932457786116
Tuning accéléré -> P10_WordChar_Hybrid_LinearSVC
Meilleurs paramètres : {'C': 2.0, 'class_weight': 'balanced'}
Meilleur score de validation : 0.7196819085487077


Le tuning s'est déroulé sans erreur sur les 5 pipelines. Voici les meilleurs paramètres trouvés :

- **P06 (Sublinear + LogReg)** : `C=2.0, class_weight='balanced'` — score de validation : **0.686**
- **P07 (Char + LogReg)** : `C=1.0, class_weight='balanced'` — score de validation : **0.690**
- **P08 (Char + LinearSVC)** : `C=4.0, class_weight=None` — score de validation : **0.697**
- **P09 (Hybride + LogReg)** : `C=4.0, class_weight='balanced'` — score de validation : **0.709**
- **P10 (Hybride + LinearSVC)** : `C=2.0, class_weight='balanced'` — score de validation : **0.720**

Comme dans le NB1, `class_weight='balanced'` est sélectionné pour 4 pipelines sur 5, confirmant que le déséquilibre des classes reste un facteur clé à corriger. **P10 obtient le meilleur score de validation**, ce qui laisse présager de bonnes performances sur le test. Fait intéressant, P08 est à nouveau le seul pipeline pour lequel le rééquilibrage n'a pas aidé — LinearSVC sur représentation caractères semble se débrouiller naturellement mieux sans ce correctif.

In [ ]:
tuning_best_df = round_results(pd.DataFrame(tuning_rows))
display(tuning_best_df)

tuned_results_df = round_results(pd.DataFrame(tuned_metrics_rows))
display(tuned_results_df)

if tuning_failures:
    print("\nPipelines tuning en échec :")
    display(pd.DataFrame(tuning_failures))


pipeline,P06_TFIDF_Sublinear_LogReg,P07_CharTFIDF_LogReg,P08_CharTFIDF_LinearSVC,P09_WordChar_Hybrid_LogReg,P10_WordChar_Hybrid_LinearSVC
classifier_name,clf,clf,clf,clf,clf
params,"{""C"": 2.0, ""class_weight"": ""balanced""}","{""C"": 1.0, ""class_weight"": ""balanced""}","{""C"": 4.0, ""class_weight"": null}","{""C"": 4.0, ""class_weight"": ""balanced""}","{""C"": 2.0, ""class_weight"": ""balanced""}"
primary_metric,f1_pos,f1_pos,f1_pos,f1_pos,f1_pos
primary_score,0.6859,0.6904,0.6972,0.7092,0.7197
val_accuracy,0.8725,0.8659,0.8886,0.8864,0.8967
val_precision_macro,0.7866,0.7784,0.8175,0.8088,0.8307
val_recall_macro,0.8245,0.8417,0.8116,0.8315,0.8257
val_f1_macro,0.8030,0.8024,0.8145,0.8193,0.8282
val_precision_weighted,0.8829,0.8870,0.8877,0.8913,0.8959
val_recall_weighted,0.8725,0.8659,0.8886,0.8864,0.8967


pipeline,P06_TFIDF_Sublinear_LogReg,P07_CharTFIDF_LogReg,P08_CharTFIDF_LinearSVC,P09_WordChar_Hybrid_LogReg,P10_WordChar_Hybrid_LinearSVC
train_accuracy,0.9686,0.9318,0.9993,0.9901,0.9982
train_precision_macro,0.9299,0.8694,0.9994,0.9747,0.9953
train_recall_macro,0.9761,0.9367,0.9985,0.9939,0.9989
train_f1_macro,0.9508,0.8972,0.9989,0.9840,0.9971
train_precision_weighted,0.9722,0.9430,0.9993,0.9906,0.9983
train_recall_weighted,0.9686,0.9318,0.9993,0.9901,0.9982
train_f1_weighted,0.9694,0.9347,0.9993,0.9902,0.9982
train_precision_class_0,0.9972,0.9865,0.9993,1.0000,1.0000
train_recall_class_0,0.9641,0.9290,0.9999,0.9878,0.9978
train_f1_class_0,0.9804,0.9569,0.9996,0.9939,0.9989


Après tuning, les résultats sur le **set de test** sont encourageants :

Sur la **classe 1 (Disaster)** :
- **P09 (Hybride + LogReg)** se démarque avec le meilleur **F1 classe 1 (0.703)**, le meilleur **F1 macro (0.816)** et la meilleure **balanced accuracy (0.840)**. C'est le pipeline le plus équilibré entre précision et rappel sur les tweets disaster.
- **P07 (Char + LogReg)** obtient le meilleur **recall classe 1 (0.806)** — il détecte le plus de vrais tweets de catastrophe — avec également la meilleure balanced accuracy sur la validation (0.842). Une bonne option si on privilégie la sensibilité.
- **P10 (Hybride + LinearSVC)** ressort avec la meilleure **précision classe 1 (0.690)** et le meilleur **PR-AUC (0.782)**, ce qui en fait un bon choix si on cherche à limiter les faux positifs.
- **P06 (Sublinear + LogReg)** et **P08 (Char + LinearSVC)** restent en retrait sur la classe 1, avec des F1 autour de 0.64-0.70.

⚠️ L'overfitting est encore visible sur P08 (train_f1_class_1 = 0.998 vs test = 0.644) et dans une moindre mesure sur P09 et P10. Dans l'ensemble, **P09 est le pipeline le plus solide de ce notebook**, avec un bon équilibre entre toutes les métriques sur le test.

In [ ]:
comparison_df = compare_baseline_vs_tuned(
    baseline_df=pd.DataFrame(resultats),
    tuned_df=pd.DataFrame(tuned_metrics_rows),
)
display(round_results(comparison_df))


,baseline_pipeline,baseline_train_accuracy,baseline_train_precision_macro,baseline_train_recall_macro,baseline_train_f1_macro,baseline_train_precision_weighted,baseline_train_recall_weighted,baseline_train_f1_weighted,baseline_train_precision_class_0,baseline_train_recall_class_0,baseline_train_f1_class_0,baseline_train_support_class_0,baseline_train_precision_class_1,baseline_train_recall_class_1,baseline_train_f1_class_1,baseline_train_support_class_1,baseline_train_balanced_accuracy,baseline_train_roc_auc,baseline_train_pr_auc,baseline_test_accuracy,baseline_test_precision_macro,baseline_test_recall_macro,baseline_test_f1_macro,baseline_test_precision_weighted,baseline_test_recall_weighted,baseline_test_f1_weighted,baseline_test_precision_class_0,baseline_test_recall_class_0,baseline_test_f1_class_0,baseline_test_support_class_0,baseline_test_precision_class_1,baseline_test_recall_class_1,baseline_test_f1_class_1,baseline_test_support_class_1,baseline_test_balanced_accuracy,baseline_test_roc_auc,baseline_test_pr_auc,tuned_pipeline,tuned_train_accuracy,tuned_train_precision_macro,tuned_train_recall_macro,tuned_train_f1_macro,tuned_train_precision_weighted,tuned_train_recall_weighted,tuned_train_f1_weighted,tuned_train_precision_class_0,tuned_train_recall_class_0,tuned_train_f1_class_0,tuned_train_support_class_0,tuned_train_precision_class_1,tuned_train_recall_class_1,tuned_train_f1_class_1,tuned_train_support_class_1,tuned_train_balanced_accuracy,tuned_train_roc_auc,tuned_train_pr_auc,tuned_test_accuracy,tuned_test_precision_macro,tuned_test_recall_macro,tuned_test_f1_macro,tuned_test_precision_weighted,tuned_test_recall_weighted,tuned_test_f1_weighted,tuned_test_precision_class_0,tuned_test_recall_class_0,tuned_test_f1_class_0,tuned_test_support_class_0,tuned_test_precision_class_1,tuned_test_recall_class_1,tuned_test_f1_class_1,tuned_test_support_class_1,tuned_test_balanced_accuracy,tuned_test_roc_auc,tuned_test_pr_auc,tuned_best_params,tuned_best_val_primary_score,tuned_best_val_f1_class_1,tuned_best_val_recall_class_1,tuned_best_val_f1_macro,tuned_best_val_balanced_accuracy,delta_test_f1_class_1,delta_test_recall_class_1,delta_test_f1_macro,delta_test_balanced_accuracy
0,P06_TFIDF_Sublinear_LogReg,0.8989,0.9372,0.7309,0.7858,0.9080,0.8989,0.8836,0.8907,0.9982,0.9414,7405.0000,0.9837,0.4636,0.6302,1691.0000,0.7309,0.9773,0.9324,0.8707,0.8822,0.6680,0.7120,0.8738,0.8707,0.8463,0.8688,0.9908,0.9258,1851.0000,0.8957,0.3452,0.4983,423.0000,0.6680,0.9165,0.7687,P06_TFIDF_Sublinear_LogReg,0.9686,0.9299,0.9761,0.9508,0.9722,0.9686,0.9694,0.9972,0.9641,0.9804,7405.0000,0.8627,0.9882,0.9212,1691.0000,0.9761,0.9957,0.9789,0.8777,0.7941,0.8346,0.8115,0.8883,0.8777,0.8817,0.9441,0.9033,0.9232,1851.0000,0.6441,0.7660,0.6998,423.0000,0.8346,0.9229,0.7748,"{""C"": 2.0, ""class_weight"": ""balanced""}",0.6859,0.6859,0.7480,0.8030,0.8245,0.2015,0.4208,0.0995,0.1666
1,P07_CharTFIDF_LogReg,0.9070,0.9341,0.7565,0.8098,0.9128,0.9070,0.8952,0.9003,0.9961,0.9458,7405.0000,0.9679,0.5169,0.6739,1691.0000,0.7565,0.9709,0.9111,0.8703,0.8567,0.6787,0.7211,0.8669,0.8703,0.8492,0.8730,0.9838,0.9251,1851.0000,0.8404,0.3735,0.5172,423.0000,0.6787,0.9087,0.7455,P07_CharTFIDF_LogReg,0.9318,0.8694,0.9367,0.8972,0.9430,0.9318,0.9347,0.9865,0.9290,0.9569,7405.0000,0.7522,0.9444,0.8374,1691.0000,0.9367,0.9829,0.9293,0.8610,0.7724,0.8399,0.7972,0.8850,0.8610,0.8686,0.9517,0.8736,0.9110,1851.0000,0.5930,0.8061,0.6834,423.0000,0.8399,0.9125,0.7434,"{""C"": 1.0, ""class_weight"": ""balanced""}",0.6904,0.6904,0.8031,0.8024,0.8417,0.1662,0.4326,0.0760,0.1612
2,P08_CharTFIDF_LinearSVC,0.9919,0.9929,0.9802,0.9864,0.9919,0.9919,0.9918,0.9913,0.9988,0.9950,7405.0000,0.9945,0.9616,0.9778,1691.0000,0.9802,0.9997,0.9986,0.8835,0.8242,0.7643,0.7887,0.8765,0.8835,0.8776,0.9075,0.9541,0.9302,1851.0000,0.7409,0.5745,0.6471,423.0000,0.7643,0.9141,0.7524,P08_CharTFIDF_LinearSVC,0.9993,0.9994,0.9985,0.9989,0.9993,0.9993,0.9993,0.9993,0.9999,0.9996,7405.0000,0.9994,0.9970,0.9982

Le tableau comparatif baseline vs tuned révèle des gains très significatifs pour ce notebook :

- **P06 (Sublinear + LogReg)** : gain de **+0.42 sur le recall classe 1** et **+0.20 sur le F1 classe 1** — le tuning avec `class_weight='balanced'` a transformé ce pipeline qui ratait presque tous les tweets disaster en baseline.
- **P07 (Char + LogReg)** : gain de **+0.43 sur le recall** et **+0.17 sur le F1 classe 1**, avec une balanced accuracy qui progresse de +0.16. La représentation caractères combinée au rééquilibrage des classes donne d'excellents résultats.
- **P08 (Char + LinearSVC)** : quasi aucun gain sur le F1 classe 1 (-0.003) mais une légère amélioration du recall (+0.03). Ce pipeline semble avoir atteint ses limites avec la représentation caractères seule.
- **P09 (Hybride + LogReg)** : bon gain de **+0.27 sur le recall** et **+0.10 sur le F1 classe 1**, avec une balanced accuracy en hausse de +0.10. La combinaison mots + caractères avec rééquilibrage tire bien son épingle du jeu.
- **P10 (Hybride + LinearSVC)** : progression modeste (+0.05 sur le recall, quasi nul sur le F1), ce pipeline était déjà le plus performant en baseline et le tuning n'a pas apporté grand-chose de plus.

En conclusion, **P06 et P07 sont les grandes surprises de ce notebook** — des pipelines moyens en baseline qui deviennent très compétitifs après tuning. **P09 reste le meilleur compromis global**, avec un bon équilibre entre toutes les métriques sur le test.

In [ ]:
pd.DataFrame(tuning_rows).to_csv(TUNING_OUTPUT_DIR / f"{OUTPUT_STEM}_tuning_resume.csv", index=False)
pd.DataFrame(tuned_metrics_rows).to_csv(TUNING_OUTPUT_DIR / f"{OUTPUT_STEM}_tuned_final_results.csv", index=False)
pd.DataFrame(tuning_failures).to_csv(TUNING_OUTPUT_DIR / f"{OUTPUT_STEM}_tuning_failures.csv", index=False)

comparison_df.to_csv(TUNING_OUTPUT_DIR / f"{OUTPUT_STEM}_baseline_vs_tuned.csv", index=False)

print("Exports tuning enregistrés dans :", TUNING_OUTPUT_DIR)


Exports tuning enregistrés dans : ..\..\outputs\NB2\tuning


Tous les résultats du tuning sont sauvegardés dans `outputs/NB2/tuning` : le résumé des combinaisons testées, les métriques finales des modèles tunés, les éventuels échecs, et le tableau comparatif baseline vs tuned. Ces fichiers serviront de référence pour analyser les performances dans les prochains notebooks.

Le notebook contient désormais les deux temps de travail : un benchmark initial sans optimisation, puis une optimisation accélérée centrée sur les algorithmes.